In [28]:
import sys, time
sys.path.append(r"C:\Users\user\AppData\Roaming\Python\Python312\site-packages")

import pandas as pd
import numpy as np

# ---------- 自动尝试：ETF -> 新浪指数 -> baostock，哪个成用哪个 ----------
def load_hs300():
    for i in range(3):
        try:
            import akshare as ak
            df = ak.fund_etf_hist_em(symbol="510300", period="daily",
                                      start_date="20180101", end_date="20260907", adjust="qfq")
            df["日期"] = pd.to_datetime(df["日期"])
            df = df.set_index("日期").sort_index()[["收盘"]].rename(columns={"收盘": "close"})
            df.to_csv("hs300.csv", encoding="utf-8-sig")
            print("成功：fund_etf_hist_em(510300)")
            return df
        except Exception as e:
            print("方案1失败:", type(e).__name__, str(e)[:80]); time.sleep(3)

    for i in range(3):
        try:
            import akshare as ak
            df = ak.stock_zh_index_daily(symbol="sh000300")
            df["date"] = pd.to_datetime(df["date"])
            df = df.set_index("date").sort_index()[["close"]].rename(columns={"close": "close"})
            df.to_csv("hs300.csv", encoding="utf-8-sig")
            print("成功：stock_zh_index_daily(sh000300)")
            return df
        except Exception as e:
            print("方案2失败:", type(e).__name__, str(e)[:80]); time.sleep(3)

    try:
        import baostock as bs
        bs.login()
        rs = bs.query_history_k_data_plus("SH.000300",
            "date,open,high,low,close,volume",
            start_date="2020-01-01", end_date="2026-09-07", frequency="d")
        rows = []
        while rs.error_code == "0" and rs.next():
            rows.append(rs.get_row_data())
        bs.logout()
        df = pd.DataFrame(rows, columns=rs.fields)
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")[["close"]].sort_index()
        df.to_csv("hs300.csv", encoding="utf-8-sig")
        print("成功：baostock(SH.000300)")
        return df
    except Exception as e:
        print("方案3失败:", type(e).__name__, str(e)[:80])

    raise RuntimeError("所有数据源都失败了，请检查网络")

df = load_hs300()
print("行数:", len(df))
print(df.head())

# ---------- 双均线回测 ----------
df["ma5"]  = df["close"].rolling(20).mean()
df["ma20"] = df["close"].rolling(60).mean()
df["signal"]   = np.where(df["ma5"] > df["ma20"], 1, 0)
df["ret"]      = df["close"].pct_change()
trade = (df["signal"].diff().fillna(0) != 0).astype(float)
fee = trade * 0.001
df["strat_ret"] = df["signal"].shift(1) * df["ret"] - fee
df["cum_market"] = (1 + df["ret"].fillna(0)).cumprod()
df["cum_strat"]  = (1 + df["strat_ret"].fillna(0)).cumprod()

ann, ann_ret = 252, df["strat_ret"].mean() * 252
vol = df["strat_ret"].std() * np.sqrt(ann)
sharpe = ann_ret / vol if vol > 0 else 0
mdd = (df["cum_strat"] / df["cum_strat"].cummax() - 1).min()

print("\n样本区间:", df.index[0].date(), "~", df.index[-1].date())
print("策略年化收益:", round(ann_ret, 4))
print("夏普比率:", round(sharpe, 4))
print("最大回撤:", round(mdd, 4))
print("策略最终净值:", round(df["cum_strat"].iloc[-1], 4))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(df.index, df["cum_market"], label="沪深300")
plt.plot(df.index, df["cum_strat"],  label="MA5/MA20策略")
plt.legend(); plt.title("Double Moving Average Backtest on HS300")
plt.tight_layout(); plt.savefig("backtest_ma_result.png", dpi=120)
print("图已保存")


方案1失败: ConnectionError ('Connection aborted.', RemoteDisconnected('Remote end closed connection without
方案1失败: ConnectionError ('Connection aborted.', RemoteDisconnected('Remote end closed connection without
方案1失败: ConnectionError ('Connection aborted.', RemoteDisconnected('Remote end closed connection without
成功：stock_zh_index_daily(sh000300)
行数: 5986
               close
date                
2002-01-04  1316.455
2002-01-07  1302.084
2002-01-08  1292.714
2002-01-09  1272.645
2002-01-10  1281.261

样本区间: 2002-01-04 ~ 2026-09-04
策略年化收益: 0.0795
夏普比率: 0.4702
最大回撤: -0.4873
策略最终净值: 4.6936


C:\Users\user\AppData\Local\Temp\ipykernel_28148\4099010716.py:88: UserWarning: Glyph 27818 (\N{CJK UNIFIED IDEOGRAPH-6CAA}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.savefig("backtest_ma_result.png", dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_28148\4099010716.py:88: UserWarning: Glyph 28145 (\N{CJK UNIFIED IDEOGRAPH-6DF1}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.savefig("backtest_ma_result.png", dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_28148\4099010716.py:88: UserWarning: Glyph 31574 (\N{CJK UNIFIED IDEOGRAPH-7B56}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.savefig("backtest_ma_result.png", dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_28148\4099010716.py:88: UserWarning: Glyph 30053 (\N{CJK UNIFIED IDEOGRAPH-7565}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.savefig("backtest_ma_result.png", dpi=120)


图已保存
